In [42]:
import torch.nn as nn
import pandas as pd
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import torch._dynamo
torch._dynamo.config.suppress_errors = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

True

In [43]:
train_df = pd.read_csv("data/asl_data/sign_mnist_train.csv")
valid_df = pd.read_csv("data/asl_data/sign_mnist_valid.csv")

In [44]:
sample_df = train_df.head().copy()  # Grab the top 5 rows
sample_df.pop('label')
sample_x = sample_df.values
sample_x

array([[107, 118, 127, ..., 204, 203, 202],
       [155, 157, 156, ..., 103, 135, 149],
       [187, 188, 188, ..., 195, 194, 195],
       [211, 211, 212, ..., 222, 229, 163],
       [164, 167, 170, ..., 163, 164, 179]], shape=(5, 784))

In [45]:
sample_x.shape

(5, 784)

In [46]:
IMG_HEIGHT = 28
IMG_WIDTH = 28
IMG_CHS = 1

sample_x = sample_x.reshape(-1, IMG_CHS, IMG_HEIGHT, IMG_WIDTH)
sample_x.shape

(5, 1, 28, 28)

In [47]:

class MyDataset(Dataset):
    def __init__(self, base_df):
        x_df = base_df.copy()  # Some operations below are in-place
        y_df = x_df.pop('label')
        x_df = x_df.values / 255  # Normalize values from 0 to 1
        x_df = x_df.reshape(-1, IMG_CHS, IMG_WIDTH, IMG_HEIGHT)
        self.xs = torch.tensor(x_df).float().to(device)
        self.ys = torch.tensor(y_df).to(device)

    def __getitem__(self, idx):
        x = self.xs[idx]
        y = self.ys[idx]
        return x, y

    def __len__(self):
        return len(self.xs)

In [48]:
BATCH_SIZE = 32

train_data = MyDataset(train_df)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
train_N = len(train_loader.dataset)

valid_data = MyDataset(valid_df)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE)
valid_N = len(valid_loader.dataset)

In [49]:
batch = next(iter(train_loader))
batch

[tensor([[[[0.1569, 0.1608, 0.1686,  ..., 0.7216, 0.7333, 0.7412],
           [0.1569, 0.1569, 0.1647,  ..., 0.7176, 0.7333, 0.7490],
           [0.1490, 0.1529, 0.1608,  ..., 0.7176, 0.7333, 0.7490],
           ...,
           [0.3176, 0.3176, 0.3137,  ..., 0.2627, 0.1961, 0.1255],
           [0.3137, 0.3137, 0.3137,  ..., 0.3255, 0.2627, 0.1882],
           [0.3137, 0.3137, 0.3098,  ..., 0.3647, 0.3137, 0.2431]]],
 
 
         [[[0.6157, 0.6275, 0.6431,  ..., 0.6118, 0.6078, 0.6000],
           [0.6157, 0.6314, 0.6471,  ..., 0.6196, 0.6157, 0.6039],
           [0.6196, 0.6353, 0.6510,  ..., 0.6275, 0.6196, 0.6157],
           ...,
           [0.6784, 0.7059, 0.7176,  ..., 0.7255, 0.7255, 0.7216],
           [0.6745, 0.6980, 0.7176,  ..., 0.7255, 0.7216, 0.7137],
           [0.6745, 0.7020, 0.7137,  ..., 0.7294, 0.7176, 0.7098]]],
 
 
         [[[0.5882, 0.6000, 0.6118,  ..., 0.6196, 0.6157, 0.6078],
           [0.5922, 0.6039, 0.6157,  ..., 0.6275, 0.6235, 0.6196],
           [0.6000

In [50]:
batch[0].shape

torch.Size([32, 1, 28, 28])

In [51]:
batch[1].shape

torch.Size([32])

In [52]:
n_classes = 24
kernel_size = 3
flattened_img_size = 75 * 3 * 3

model = nn.Sequential(
    # First convolution
    nn.Conv2d(IMG_CHS, 25, kernel_size, stride=1, padding=1),  # 25 x 28 x 28
    nn.BatchNorm2d(25),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # 25 x 14 x 14
    # Second convolution
    nn.Conv2d(25, 50, kernel_size, stride=1, padding=1),  # 50 x 14 x 14
    nn.BatchNorm2d(50),
    nn.ReLU(),
    nn.Dropout(.2),
    nn.MaxPool2d(2, stride=2),  # 50 x 7 x 7
    # Third convolution
    nn.Conv2d(50, 75, kernel_size, stride=1, padding=1),  # 75 x 7 x 7
    nn.BatchNorm2d(75),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # 75 x 3 x 3
    # Flatten to Dense
    nn.Flatten(),
    nn.Linear(flattened_img_size, 512),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(512, n_classes)
)

In [53]:
try:
    model = torch.compile(model.to(device))
except:
    model = model.to(device)

In [54]:
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters())

In [55]:
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N

In [56]:

def validate():
    loss = 0
    accuracy = 0

    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            output = model(x)

            loss += loss_function(output, y).item()
            accuracy += get_batch_accuracy(output, y, valid_N)
    print('Valid - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [57]:

def train():
    loss = 0
    accuracy = 0

    model.train()
    for x, y in train_loader:
        output = model(x)
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)
    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [58]:
epochs = 20

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train()
    validate()

Epoch: 0


W0531 14:56:13.838000 25828 site-packages\torch\_dynamo\convert_frame.py:1125] WON'T CONVERT inner c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\_dynamo\external_utils.py line 38 
W0531 14:56:13.838000 25828 site-packages\torch\_dynamo\convert_frame.py:1125] due to: 
W0531 14:56:13.838000 25828 site-packages\torch\_dynamo\convert_frame.py:1125] Traceback (most recent call last):
W0531 14:56:13.838000 25828 site-packages\torch\_dynamo\convert_frame.py:1125]   File "c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\_dynamo\output_graph.py", line 1446, in _call_user_compiler
W0531 14:56:13.838000 25828 site-packages\torch\_dynamo\convert_frame.py:1125]     compiled_fn = compiler_fn(gm, self.example_inputs())
W0531 14:56:13.838000 25828 site-packages\torch\_dynamo\convert_frame.py:1125]                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
W0531 14:56:13.838000 25828 site-packages\torch\_dynamo\convert_frame.py:1125]   File "c:

Train - Loss: 257.3031 Accuracy: 0.9119
Valid - Loss: 19.1595 Accuracy: 0.9685
Epoch: 1
Train - Loss: 14.6081 Accuracy: 0.9959
Valid - Loss: 47.3073 Accuracy: 0.9325
Epoch: 2
Train - Loss: 10.0816 Accuracy: 0.9968
Valid - Loss: 18.0416 Accuracy: 0.9731
Epoch: 3
Train - Loss: 12.5533 Accuracy: 0.9957
Valid - Loss: 22.3468 Accuracy: 0.9670
Epoch: 4
Train - Loss: 5.9998 Accuracy: 0.9983
Valid - Loss: 27.3418 Accuracy: 0.9584
Epoch: 5
Train - Loss: 12.6347 Accuracy: 0.9955
Valid - Loss: 107.2497 Accuracy: 0.8904
Epoch: 6
Train - Loss: 7.3974 Accuracy: 0.9976
Valid - Loss: 15.5056 Accuracy: 0.9727
Epoch: 7
Train - Loss: 2.9298 Accuracy: 0.9992
Valid - Loss: 22.5029 Accuracy: 0.9649
Epoch: 8
Train - Loss: 8.4049 Accuracy: 0.9973
Valid - Loss: 15.6781 Accuracy: 0.9755
Epoch: 9
Train - Loss: 7.2196 Accuracy: 0.9976
Valid - Loss: 25.7427 Accuracy: 0.9721
Epoch: 10
Train - Loss: 4.6875 Accuracy: 0.9985
Valid - Loss: 24.0302 Accuracy: 0.9693
Epoch: 11
Train - Loss: 4.6895 Accuracy: 0.9984
Valid -